# Extract NASA reference clauses

Builds `data/clauses.json`, the reference corpus the SRS review retrieves from.

| Source | File | Authority |
|---|---|---|
| NPR 7150.2D | `data/sources/N_PR_7150_002D_.pdf` | mandatory |
| SWE Handbook 5.09 | `data/sources/swehb_5.09_srs.html` | recommended practice |

Each chunk keeps its source, section, SWE id, source URL and a passage `type`:

| `type` | Meaning | Source |
|---|---|---|
| `requirement` | a binding *shall* statement tagged `[SWE-xxx]` | NPR |
| `note` | an explanatory Note attached to a requirement - **not binding** | NPR |
| `explanatory` | untagged context text in a section | NPR |
| `guidance` | Handbook recommended practice - never binding | SWEHB |

The distinction matters when citing: the Note under 4.1.2 describes what a good
requirement is (well-formed, complete, conflict-free, verifiable), but it is
explanatory text, not a NASA requirement.

In [1]:
%pip install -q pypdf requests beautifulsoup4 lxml pandas


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import re, json
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader

# Resolve the repo root so this runs from the root or from notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "sources").is_dir())
SOURCES = ROOT / "data" / "sources"

NPR_PDF = SOURCES / "N_PR_7150_002D_.pdf"
SWEHB_HTML = SOURCES / "swehb_5.09_srs.html"
OUT = ROOT / "data" / "clauses.json"

SWEHB_URL = ("https://swehb.nasa.gov/spaces/SWEHBVD/pages/102695669/"
             "5.09+-+SRS+-+Software+Requirements+Specification")
NPR_CHAPTER_URL = ("https://nodis3.gsfc.nasa.gov/displayDir.cfm"
                   "?Internal_ID=N_PR_7150_002D_&page_name=Chapter{}")

## NPR 7150.2D

### Text extraction

Every page ends with the same furniture: a disclaimer, `Page  26  of  89`, and a
running head `NPR 7150.2D -- Chapter3 Page  26  of  89`. The spacing is irregular
(note the double spaces), so every gap in these patterns is `\s+`.

Stripping happens **per page, before the pages are joined**. On the joined text, a
pattern like `NPR 7150.2D --.*?Page` can run from one page's header to a later
page's footer and delete the body in between.

In [3]:
_DISCLAIMER = re.compile(
    r"This\s+document\s+does\s+not\s+bind\s+the\s+public.*?nodis3\.gsfc\.nasa\.gov\.?",
    re.S | re.I)
_RUNNING_HEAD = re.compile(r"NPR\s+7150\.2D\s*--\s*\w+\s*Page\s+\d+\s+of\s+\d+", re.I)
_PAGE_NO = re.compile(r"^\s*Page\s+\d+\s+of\s+\d+\s*$", re.M | re.I)


def npr_text(path=NPR_PDF):
    pages = []
    for page in PdfReader(path).pages:
        t = page.extract_text() or ""
        t = _DISCLAIMER.sub(" ", t)
        t = _RUNNING_HEAD.sub(" ", t)
        t = _PAGE_NO.sub(" ", t)
        pages.append(t)
    return re.sub(r"\s+", " ", "\n".join(pages)).strip()

### Trim to Chapters 1-6

Everything before the second "Chapter 1. Introduction" is the table of contents,
which yields dozens of heading-only fragments. Everything from "Appendix A.
Definitions" on includes Appendix C, a requirements mapping matrix that restates
every clause in a different format (`4.1.2 050 The project manager shall...`) and
would double-count the document.

In [4]:
def npr_body(text):
    starts = [m.start() for m in re.finditer(r"Chapter\s*1\.\s+Introduction", text, re.I)]
    ends = [m.start() for m in re.finditer(r"Appendix\s+A\.\s+Definitions", text, re.I)]
    lo = starts[-1] if len(starts) > 1 else 0
    hi = ends[-1] if ends else len(text)
    return text[lo:hi]

### Chunk by section

Chunks are cut at section numbers (`3.7.3`), not at `[SWE-xxx]` tags. In the NPR
the tag often sits *before* a list:

> 3.7.3 ...the project manager shall implement the following items in the
> software: **[SWE-134]** a. The software is initialized... b. ... l. ...

Cutting at the tag would keep the lead-in and drop items a-l. Cutting at the next
section number keeps the whole clause together. Notes are split into their own
`note` chunks, linked to the same section and SWE id.

In [5]:
# "3.7.3 If a project..." - a single leading digit keeps document numbers such as
# NASA-STD-8739.8 and NPR 7120.5 from being read as section headings.
_SECTION = re.compile(r"(?:(?<=\s)|^)([1-9]\.\d+(?:\.\d+)*)\s+(?=[A-Z“\"(])")
_SWE = re.compile(r"\[SWE-(\d{3})\]")
_NOTE = re.compile(r"\bNote:\s*", re.I)


def _npr_row(section, swe, kind, text):
    return {
        "swe_id": f"SWE-{swe}" if swe else None,
        "section": section,
        "text": _SWE.sub("", text).strip(),
        "source": "NPR 7150.2D",
        "source_url": NPR_CHAPTER_URL.format(section.split(".")[0]),
        "type": kind,
    }


def parse_npr(path=NPR_PDF):
    text = npr_body(npr_text(path))
    marks = list(_SECTION.finditer(text))
    rows = []
    for i, m in enumerate(marks):
        section = m.group(1)
        end = marks[i + 1].start() if i + 1 < len(marks) else len(text)
        body = text[m.end():end].strip()
        if len(body) < 60 and not _SWE.search(body):
            continue  # heading-only fragment

        notes = list(_NOTE.finditer(body))
        main = body[:notes[0].start()].strip() if notes else body
        tag = _SWE.search(main)
        swe = tag.group(1) if tag else None

        if main:
            rows.append(_npr_row(section, swe, "requirement" if swe else "explanatory", main))
        for j, n in enumerate(notes):
            n_end = notes[j + 1].start() if j + 1 < len(notes) else len(body)
            note = body[n.end():n_end].strip()
            if note:
                rows.append(_npr_row(section, swe, "note", note))
    return rows

## SWE Handbook 5.09

The page is split into seven tabs. All seven bodies are in the page HTML (only
one is visible at a time), so the saved snapshot is parsed rather than a printed
PDF, which captures only the open tab.

Tab 1 numbers its content with letters (`a. Introduction`, `c. Qualification
Provisions`) while tab 3 uses digits (`3.2.1.1`), so both heading styles are
matched. Letter headings are qualified with their tab number: `1.c`.

`fetch_swehb_snapshot()` refreshes the saved copy from the live site. It is only
called if the snapshot is missing.

In [6]:
_H_NUM = re.compile(r"(?:(?<=\s)|^)(\d+(?:\.\d+)+)\s+(?=[A-Z])")
_H_ALPHA = re.compile(r"(?:(?<=\s)|^)([a-z])\.\s+(?=[A-Z])")


def fetch_swehb_snapshot(path=SWEHB_HTML, url=SWEHB_URL):
    r = requests.get(url, timeout=45, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    path.write_text(r.text, encoding="utf-8")
    return path


def _swehb_row(section, text, tab):
    return {
        "swe_id": None,
        "section": section,
        "text": text,
        "source": "SWEHB 5.09",
        "source_url": f"{SWEHB_URL}#tabs-{tab}",
        "type": "guidance",
    }


def parse_swehb(path=SWEHB_HTML):
    soup = BeautifulSoup(path.read_text(encoding="utf-8"), "lxml")
    rows = []
    for tab in range(1, 8):
        div = soup.select_one(f"#tabs-{tab}")
        if div is None:
            continue
        body = re.sub(r"\s+", " ", div.get_text(" ", strip=True))
        marks = sorted([*_H_NUM.finditer(body), *_H_ALPHA.finditer(body)],
                       key=lambda m: m.start())
        if not marks:
            rows.append(_swehb_row(str(tab), body, tab))
            continue

        lead = body[:marks[0].start()].strip()
        if len(lead) > 40:
            rows.append(_swehb_row(str(tab), lead, tab))
        for i, m in enumerate(marks):
            end = marks[i + 1].start() if i + 1 < len(marks) else len(body)
            chunk = body[m.end():end].strip()
            if chunk:
                label = m.group(1)
                rows.append(_swehb_row(label if "." in label else f"{tab}.{label}", chunk, tab))
    return rows

## Build

In [7]:
if not SWEHB_HTML.exists():
    fetch_swehb_snapshot()

rows = parse_npr() + parse_swehb()
OUT.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding="utf-8")

df = pd.DataFrame(rows)
print(f"{len(df)} chunks -> {OUT.relative_to(ROOT)}")
df.groupby(["source", "type"]).size().rename("chunks").reset_index()

331 chunks -> data/clauses.json


,source,type,chunks
0,NPR 7150.2D,explanatory,63
1,NPR 7150.2D,note,56
2,NPR 7150.2D,requirement,130
3,SWEHB 5.09,guidance,82


In [8]:
df.head(10)

,swe_id,section,text,source,source_url,type
0,NaN,1.2,Hierarchy of NASA Software-Related Engineering...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...,explanatory
1,NaN,1.3,"Document Structure Chapter 2. Roles, Responsib...",NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...,explanatory
2,NaN,2.2,Principles Related to Tailoring of the Require...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...,explanatory
3,NaN,3.6,Software Assurance and Software Independent Ve...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...,explanatory
4,NaN,3.12,Software Bi-Directional Traceability Chapter 4...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...,explanatory
5,NaN,4.6,"Software Operations, Maintenance, and Retireme...",NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...,explanatory
6,NaN,5.5,Software Non-conformance or Defect Management ...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...,explanatory
7,NaN,6.2,Software Engineering Product Content Appendix ...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...,explanatory
8,NaN,6.2,The above statement alone is not sufficient to...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...,note
9,NaN,1.1.1,This directive imposes requirements on procedu...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...,explanatory


## Checks

Regression checks on extraction quality. The notebook fails here if any of these
break after a change.

In [9]:
checks = {
    "all 130 SWE clauses present":
        df["swe_id"].dropna().nunique() == 130,
    "no page furniture in text":
        not df["text"].str.contains(r"Page\s+\d+\s+of\s+89|This document does not bind", regex=True).any(),
    "lists stay attached (SWE-134 items a-l)":
        df.loc[(df.swe_id == "SWE-134") & (df.type == "requirement"), "text"]
          .str.contains("two independent actions").any(),
    "notes kept and labelled (4.1.2)":
        df.loc[df.type == "note", "text"].str.contains("well-formed").any(),
    "Handbook tab 1 present":
        df.loc[df.source == "SWEHB 5.09", "section"].str.startswith("1.").any(),
    "Handbook tab 1 content (qualification methods)":
        df["text"].str.contains("Demonstration, Testing, Analysis, Inspection").any(),
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)
assert all(checks.values()), "extraction check failed"

PASS all 130 SWE clauses present
PASS no page furniture in text
PASS lists stay attached (SWE-134 items a-l)
PASS notes kept and labelled (4.1.2)
PASS Handbook tab 1 present
PASS Handbook tab 1 content (qualification methods)


Section 4.1, the NPR section the SRS review is scoped to:

In [10]:
df.loc[(df.source == "NPR 7150.2D") & df.section.str.startswith("4.1"),
       ["section", "swe_id", "type", "text"]].assign(text=lambda d: d.text.str[:80])

,section,swe_id,type,text
161,4.1.1,NaN,explanatory,The requirements phase is one of the most crit...
162,4.1.2,SWE-050,requirement,"The project manager shall establish, capture, ..."
163,4.1.2,SWE-050,note,The software technical requirements definition...
164,4.1.3,SWE-051,requirement,The project manager shall perform software req...
165,4.1.4,SWE-184,requirement,The project manager shall include software rel...
166,4.1.5,SWE-053,requirement,The project manager shall track and manage cha...
167,4.1.6,SWE-054,requirement,"The project manager shall identify, initiate c..."
168,4.1.7,SWE-055,requirement,The project manager shall perform requirements...
